In [1]:
# ============================================================
# IoT Indoor Air Quality Prediction
# FINAL THESIS VERSION (Leak-Free + Ablation + Audits)
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import os
import re
import random
import logging
import joblib

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

# ============================================================
# SKLEARN
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.impute import SimpleImputer

from sklearn.model_selection import (
    TimeSeriesSplit,
    RandomizedSearchCV,
    cross_val_score
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.covariance import EllipticEnvelope
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)
from sklearn.feature_selection import SelectKBest, f_regression

# ============================================================
# OPTIONAL LIBRARIES
# ============================================================

try:
    from xgboost import XGBRegressor
    XGB_AVAILABLE = True
except:
    XGB_AVAILABLE = False

try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    SARIMA_AVAILABLE = True
except:
    SARIMA_AVAILABLE = False

try:
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping
    from sklearn.preprocessing import MinMaxScaler
    LSTM_AVAILABLE = True
except:
    LSTM_AVAILABLE = False

# ============================================================
# CONFIGURATION
# ============================================================

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s"
)

logger = logging.getLogger(__name__)

# ============================================================
# DATA PATH
# ============================================================

DATA_PATH = os.environ.get("IAQ_D2_PATH", r"C:\Users\Mero\OneDrive\Documents\data collection\device2_air_quality_2024-09-25_to_2025-04-07.csv")

# ============================================================
# LOAD DATA
# ============================================================

print("="*80)
print("Loading Dataset")
print("="*80)

df = pd.read_csv(DATA_PATH)
print(df.head())
print()
print("Dataset Shape :", df.shape)

# ============================================================
# TIMESTAMP
# ============================================================

df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df = df.sort_values("Timestamp").reset_index(drop=True)

# ============================================================
# ROBUST COLUMN DETECTION
# ============================================================

def normalize(name):
    name = name.lower()
    name = re.sub(r"[^a-z0-9]", "", name)
    return name

normalized = {c: normalize(c) for c in df.columns}

def detect_column(include, exclude=None):
    for original, norm in normalized.items():
        if exclude:
            if any(e in norm for e in exclude):
                continue
        if all(i in norm for i in include):
            return original
    return None

CO2_COL  = detect_column(["co2"])
TEMP_COL = detect_column(["temperature"])
HUM_COL  = detect_column(["humidity"])
OCC_COL  = detect_column(["occupancy"])
PM25_COL = detect_column(["pm2","5"], exclude=["ppm"])
TVOC_COL = detect_column(["tvoc"])

print()
print("Detected Columns")
print("---------------------------")
print("CO2        :", CO2_COL)
print("Temperature:", TEMP_COL)
print("Humidity   :", HUM_COL)
print("Occupancy  :", OCC_COL)
print("PM2.5      :", PM25_COL)
print("TVOC       :", TVOC_COL)

# ============================================================
# TIME FEATURES
# ============================================================

print()
print("Creating Time Features...")

df["Hour"] = df["Timestamp"].dt.hour
df["Day"] = df["Timestamp"].dt.day
df["Weekday"] = df["Timestamp"].dt.dayofweek
df["Month"] = df["Timestamp"].dt.month
df["Weekend"] = (df["Weekday"] >= 5).astype(int)

df["Hour_sin"] = np.sin(2 * np.pi * df["Hour"] / 24)
df["Hour_cos"] = np.cos(2 * np.pi * df["Hour"] / 24)

# ============================================================
# SENSOR INTERACTION FEATURES
# ============================================================

print()
print("Creating Sensor Interaction Features...")

if TEMP_COL and HUM_COL:
    df["Temp_Humidity"] = df[TEMP_COL] * df[HUM_COL]
if OCC_COL:
    df["Occupancy_CO2"] = df[OCC_COL]
if PM25_COL:
    df["PM25_CO2"] = df[PM25_COL]
if TVOC_COL:
    df["TVOC_CO2"] = df[TVOC_COL]

# ============================================================
# LEAK-FREE LAG FEATURES
# ============================================================

print()
print("Creating Lag Features...")

LAGS = [1, 2, 3, 6, 12, 24]
for lag in LAGS:
    df[f"CO2_Lag_{lag}"] = df[CO2_COL].shift(lag)

# ============================================================
# LEAK-FREE ROLLING FEATURES
# ============================================================

print()
print("Creating Rolling Features...")

history = df[CO2_COL].shift(1)
WINDOWS = [3, 6, 12, 24]

for window in WINDOWS:
    df[f"RollingMean_{window}"] = history.rolling(window).mean()
    df[f"RollingStd_{window}"] = history.rolling(window).std()
    df[f"RollingMin_{window}"] = history.rolling(window).min()
    df[f"RollingMax_{window}"] = history.rolling(window).max()

# ============================================================
# EXPONENTIAL MOVING AVERAGES
# ============================================================

print()
print("Creating EMA Features...")

df["EMA_6"] = history.ewm(span=6).mean()
df["EMA_12"] = history.ewm(span=12).mean()
df["EMA_24"] = history.ewm(span=24).mean()

# ============================================================
# FUTURE TARGET
# ============================================================

FORECAST_HORIZON = 1
df["Target"] = df[CO2_COL].shift(-FORECAST_HORIZON)

# ============================================================
# REMOVE NAN
# ============================================================

df = df.dropna().reset_index(drop=True)

print()
print("Dataset after Feature Engineering")
print(df.shape)

# ============================================================
# CHRONOLOGICAL TRAIN TEST SPLIT
# ============================================================

split_index = int(len(df) * 0.80)
train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

print()
print("="*80)
print("Training Samples :", len(train_df))
print("Testing Samples  :", len(test_df))
print("="*80)

logger.info("Part 1 completed successfully.")

# ============================================================
# PART 2: DATA PREPROCESSING
# ============================================================

print("="*80)
print("PART 2 - PREPROCESSING")
print("="*80)

X_train = train_df.drop(columns=["Target", "Timestamp"])
y_train = train_df["Target"]
X_test = test_df.drop(columns=["Target", "Timestamp"])
y_test = test_df["Target"]

# Remove Outliers (Training Data Only)
print()
print("Removing Outliers...")
numeric_cols = X_train.select_dtypes(include=np.number).columns
outlier_detector = EllipticEnvelope(contamination=0.01, random_state=RANDOM_STATE)
mask = outlier_detector.fit_predict(X_train[numeric_cols]) == 1

X_train = X_train.loc[mask].reset_index(drop=True)
y_train = y_train.loc[mask].reset_index(drop=True)

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

linear_selector = SelectKBest(score_func=f_regression, k=min(25, len(numeric_features)))
tscv = TimeSeriesSplit(n_splits=5)

results = []

def evaluate_model(model_name, model):
    model.fit(X_train, y_train)
    prediction = model.predict(X_test)

    mae = mean_absolute_error(y_test, prediction)
    rmse = np.sqrt(mean_squared_error(y_test, prediction))
    r2 = r2_score(y_test, prediction)

    cv_scores = cross_val_score(model, X_train, y_train, cv=tscv, scoring="r2", n_jobs=-1)

    print()
    print("="*60)
    print(model_name)
    print("="*60)
    print(f"MAE  : {mae:.2f}")
    print(f"RMSE : {rmse:.2f}")
    print(f"R²   : {r2:.4f}")
    print(f"CV   : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

    results.append({
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "CV Mean": cv_scores.mean(),
        "CV Std": cv_scores.std()
    })
    return model

# ============================================================
# PART 3: BASELINE + LINEAR REGRESSION + DECISION TREE
# ============================================================

print("=" * 80)
print("PART 3")
print("=" * 80)

linear_pipeline = Pipeline([
    ("prep", preprocessor),
    ("select", linear_selector),
    ("model", LinearRegression())
])

decision_tree_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", DecisionTreeRegressor(max_depth=12, min_samples_leaf=5, random_state=RANDOM_STATE))
])

# Lag-1 Persistence Baseline
lag_prediction = test_df[CO2_COL].values
lag_actual = y_test.values
lag_mae = mean_absolute_error(lag_actual, lag_prediction)
lag_rmse = np.sqrt(mean_squared_error(lag_actual, lag_prediction))
lag_r2 = r2_score(lag_actual, lag_prediction)

results.append({
    "Model": "Lag-1 Persistence",
    "MAE": lag_mae,
    "RMSE": lag_rmse,
    "R2": lag_r2,
    "CV Mean": np.nan,
    "CV Std": np.nan
})

linear_model = evaluate_model("Linear Regression", linear_pipeline)
decision_tree_model = evaluate_model("Decision Tree", decision_tree_pipeline)

# ============================================================
# PART 4: TREE-BASED MODELS
# ============================================================

print("="*80)
print("PART 4")
print("="*80)

rf_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestRegressor(n_estimators=300, max_depth=15, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1))
])

random_forest_model = evaluate_model("Random Forest", rf_pipeline)

# Tuned RF
parameter_grid = {
    "model__n_estimators": [150, 250],
    "model__max_depth": [10, 15, None],
    "model__min_samples_leaf": [1, 2],
    "model__min_samples_split": [2, 5],
    "model__max_features": ["sqrt"]
}

rf_base = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1))
])

tuned_rf = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=parameter_grid,
    n_iter=8,
    cv=tscv,
    scoring="r2",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)
tuned_rf.fit(X_train, y_train)
best_rf = tuned_rf.best_estimator_

pred_trf = best_rf.predict(X_test)
trf_mae = mean_absolute_error(y_test, pred_trf)
trf_rmse = np.sqrt(mean_squared_error(y_test, pred_trf))
trf_r2 = r2_score(y_test, pred_trf)

results.append({
    "Model": "Tuned RF",
    "MAE": trf_mae,
    "RMSE": trf_rmse,
    "R2": trf_r2,
    "CV Mean": tuned_rf.best_score_,
    "CV Std": np.nan
})

gb_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=5, subsample=0.8, random_state=RANDOM_STATE))
])
gradient_boosting_model = evaluate_model("Gradient Boosting", gb_pipeline)

if XGB_AVAILABLE:
    xgb_pipeline = Pipeline([
        ("prep", preprocessor),
        ("model", XGBRegressor(objective="reg:squarederror", n_estimators=300, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE, n_jobs=-1))
    ])
    xgb_model = evaluate_model("XGBoost", xgb_pipeline)

# ============================================================
# PART 5: SARIMA + LSTM
# ============================================================

if SARIMA_AVAILABLE:
    train_series = train_df[CO2_COL]
    test_series = test_df["Target"]
    sarima = SARIMAX(train_series, order=(2,1,2), seasonal_order=(0,0,0,0), enforce_stationarity=False, enforce_invertibility=False)
    sarima_model = sarima.fit(disp=False)
    sarima_prediction = sarima_model.forecast(steps=len(test_series))
    
    s_mae = mean_absolute_error(test_series, sarima_prediction)
    s_rmse = np.sqrt(mean_squared_error(test_series, sarima_prediction))
    s_r2 = r2_score(test_series, sarima_prediction)
    
    results.append({
        "Model": "SARIMA",
        "MAE": s_mae,
        "RMSE": s_rmse,
        "R2": s_r2,
        "CV Mean": np.nan,
        "CV Std": np.nan
    })

if LSTM_AVAILABLE:
    X_train_lstm = preprocessor.fit_transform(X_train)
    X_test_lstm = preprocessor.transform(X_test)
    if hasattr(X_train_lstm, "toarray"):
        X_train_lstm = X_train_lstm.toarray()
        X_test_lstm = X_test_lstm.toarray()

    feature_scaler = MinMaxScaler()
    target_scaler = MinMaxScaler()
    X_train_scaled = feature_scaler.fit_transform(X_train_lstm)
    X_test_scaled = feature_scaler.transform(X_test_lstm)
    y_train_scaled = target_scaler.fit_transform(y_train.values.reshape(-1,1))
    y_test_scaled = target_scaler.transform(y_test.values.reshape(-1,1))

    WINDOW_LSTM = 24
    def make_sequences(X, y, window):
        xs, ys = [], []
        for i in range(len(X) - window):
            xs.append(X[i:i+window])
            ys.append(y[i+window])
        return np.array(xs), np.array(ys)

    X_seq_train, y_seq_train = make_sequences(X_train_scaled, y_train_scaled, WINDOW_LSTM)
    X_seq_test, y_seq_test = make_sequences(X_test_scaled, y_test_scaled, WINDOW_LSTM)

    lstm_model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(WINDOW_LSTM, X_seq_train.shape[2])),
        Dropout(0.2),
        LSTM(32),
        Dense(1)
    ])
    lstm_model.compile(optimizer="adam", loss="mse")
    early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    
    lstm_model.fit(X_seq_train, y_seq_train, validation_split=0.10, epochs=30, batch_size=32, callbacks=[early_stop], verbose=0)
    
    pred_scaled = lstm_model.predict(X_seq_test, verbose=0)
    lstm_pred = target_scaler.inverse_transform(pred_scaled).flatten()
    lstm_actual = target_scaler.inverse_transform(y_seq_test).flatten()

    l_mae = mean_absolute_error(lstm_actual, lstm_pred)
    l_rmse = np.sqrt(mean_squared_error(lstm_actual, lstm_pred))
    l_r2 = r2_score(lstm_actual, lstm_pred)

    results.append({
        "Model": "LSTM",
        "MAE": l_mae,
        "RMSE": l_rmse,
        "R2": l_r2,
        "CV Mean": np.nan,
        "CV Std": np.nan
    })

# ============================================================
# PART 6: FINAL EVALUATION & EXPORTS
# ============================================================

print("="*90)
print("FINAL MODEL COMPARISON")
print("="*90)

results_df = pd.DataFrame(results).sort_values(by="R2", ascending=False).reset_index(drop=True)
print(results_df)

results_df.to_csv("Model_Comparison.csv", index=False)

# ============================================================
# PART 7: ABLATION STUDY (WITHOUT LAG & ROLLING FEATURES)
# ============================================================

print("="*80)
print("PART 7 - ABLATION STUDY (Excluding Lags & Rolling Features)")
print("="*80)

exclude_keywords = ["lag", "rolling", "ema"]
abl_numeric_features = [
    col for col in numeric_features 
    if not any(kw in col.lower() for kw in exclude_keywords)
]

print(f"Original Numeric Features Count: {len(numeric_features)}")
print(f"Features Count without Lags/Rolling: {len(abl_numeric_features)}")

abl_numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

abl_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", abl_numeric_pipeline, abl_numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

abl_rf_pipeline = Pipeline([
    ("prep", abl_preprocessor),
    ("model", RandomForestRegressor(n_estimators=300, max_depth=15, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1))
])

print("\nRunning Ablation Random Forest (Without Lags/Rolling)...")
evaluate_model("Ablation RF (No Lags/Roll)", abl_rf_pipeline)


# ============================================================
# PART 9: SIGNIFICANCE TESTS FROM THE SAME RUN AS TABLE 8
#   - per-fold R2 for every model (same TimeSeriesSplit as Part 3), paired t-tests (df = 4), Holm-adjusted
#   - Diebold-Mariano tests on the SAME test-set errors that produced Table 8 (h = 1, HLN correction; Newey-West check)
#   - provenance: data file hash, library versions, row counts, random seed
#   Nothing here is typed in by hand: every number is computed from the fitted models in this session.
#   Sign convention: for "A vs B", t = mean(R2_A - R2_B) / SE   and   DM > 0 means A has the LARGER loss (B is better).
# ============================================================
import json, hashlib, platform, itertools
import sklearn, scipy
from scipy import stats
from sklearn.base import clone

_rep = []
def rep(*a):
    s = " ".join(str(x) for x in a); print(s); _rep.append(s)

rep("="*90); rep("PART 9 - TESTS FROM THE SAME RUN AS TABLE 8"); rep("="*90)

# ---------------- provenance
def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()
meta = {"run_time": pd.Timestamp.now().isoformat(), "data_file": os.path.basename(DATA_PATH), "data_sha256": _sha256(DATA_PATH),
        "n_train_after_outlier_removal": int(len(X_train)), "n_test": int(len(X_test)), "random_state": RANDOM_STATE,
        "python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "scikit_learn": sklearn.__version__,
        "scipy": scipy.__version__, "tuned_rf_best_params": {k: (None if v is None else (v if isinstance(v, (int, float, str)) else str(v))) for k, v in tuned_rf.best_params_.items()}}
try:
    import xgboost; meta["xgboost"] = xgboost.__version__
except Exception: pass
json.dump(meta, open("d2_run_meta.json", "w"), indent=2); rep("provenance:", json.dumps({k: meta[k] for k in ("data_file", "data_sha256", "n_train_after_outlier_removal", "n_test")}))

# ---------------- fitted models from this session
fitted = {"Linear Regression": linear_model, "Decision Tree": decision_tree_model, "Random Forest (baseline)": random_forest_model,
          "Tuned RF": best_rf, "Gradient Boosting": gradient_boosting_model}
if XGB_AVAILABLE: fitted["XGBoost"] = xgb_model
table_name = {"Linear Regression": "Linear Regression", "Decision Tree": "Decision Tree", "Random Forest (baseline)": "Random Forest",
              "Tuned RF": "Tuned RF", "Gradient Boosting": "Gradient Boosting", "XGBoost": "XGBoost"}

# ---------------- A) per-fold R2 (identical folds: tscv on the outlier-filtered training set)
fold_r2 = {n: cross_val_score(clone(m), X_train, y_train, cv=tscv, scoring="r2", n_jobs=-1) for n, m in fitted.items()}
fold_df = pd.DataFrame(fold_r2); fold_df.index = [f"fold {i+1}" for i in range(len(fold_df))]
rep("\nPer-fold R2 (5 expanding-window folds)"); rep(fold_df.round(4).to_string())
chk = results_df.set_index("Model")["CV Mean"]
for n in fold_r2:
    logged = chk.get(table_name[n], np.nan)
    rep(f"  reproduction check {n:26s} fold mean = {fold_r2[n].mean():.6f} | CV Mean in Table 8 = {logged:.6f} | "
        + ("OK" if abs(fold_r2[n].mean() - logged) < 1e-6 else "MISMATCH - investigate"))
fold_df.to_csv("d2_fold_r2.csv")

def holm(p):
    p = np.asarray(p, float); order = np.argsort(p); m = len(p); adj = np.empty(m); run = 0.0
    for rank, i in enumerate(order): run = max(run, (m - rank) * p[i]); adj[i] = min(1.0, run)
    return adj

rows = []
for a, b in itertools.combinations(fold_r2, 2):
    t, p = stats.ttest_rel(fold_r2[a], fold_r2[b])
    try: w = stats.wilcoxon(fold_r2[a], fold_r2[b]).pvalue
    except Exception: w = np.nan
    rows.append({"test": "paired t (5 folds, df=4)", "A": a, "B": b, "mean_diff_R2_A_minus_B": float((fold_r2[a] - fold_r2[b]).mean()),
                 "stat": t, "p": p, "wilcoxon_p": w, "folds_where_A_better": int((fold_r2[a] > fold_r2[b]).sum())})
T = pd.DataFrame(rows); T["p_holm"] = holm(T["p"])

# ---------------- B) Diebold-Mariano on the test errors behind Table 8
yte = y_test.values
preds = {n: m.predict(X_test) for n, m in fitted.items()}
preds["Lag-1 Persistence (rule)"] = test_df[CO2_COL].values
offset = 0
if "lstm_pred" in globals():
    offset = WINDOW_LSTM                       # LSTM predicts test rows WINDOW_LSTM .. end only
    lstm_full = np.full(len(yte), np.nan); lstm_full[offset:] = lstm_pred; preds["LSTM"] = lstm_full
    if not np.allclose(lstm_actual, yte[offset:], atol=1e-3):
        rep("WARNING: LSTM targets are not aligned with y_test; LSTM is left out of the Diebold-Mariano tests"); preds.pop("LSTM")
def _dm(e1, e2, loss="se", lags=12):
    d = (e1**2 - e2**2) if loss == "se" else (np.abs(e1) - np.abs(e2)); n = len(d); db = d.mean(); dc = d - db
    g0 = np.mean(dc*dc); hln = db/np.sqrt(g0/n) * np.sqrt((n - 1)/n)                       # h = 1 -> long-run variance = gamma0
    lr = g0 + 2*sum((1 - k/(lags+1))*np.mean(dc[k:]*dc[:-k]) for k in range(1, lags+1)); nw = db/np.sqrt(lr/n)
    return hln, 2*stats.t.sf(abs(hln), n-1), nw, 2*stats.norm.sf(abs(nw)), db, n
dm_rows = []
for a, b in itertools.combinations(preds, 2):
    ok = ~(np.isnan(preds[a]) | np.isnan(preds[b])); ea, eb = (yte - preds[a])[ok], (yte - preds[b])[ok]
    s, p, snw, pnw, db, n = _dm(ea, eb, "se"); sa, pa, *_ = _dm(ea, eb, "ae")
    dm_rows.append({"test": "Diebold-Mariano (h=1, HLN)", "A": a, "B": b, "n_test_rows": n, "mean_sq_loss_diff_A_minus_B": db,
                    "stat": s, "p": p, "stat_NeweyWest12": snw, "p_NeweyWest12": pnw, "stat_abs_loss": sa, "p_abs_loss": pa,
                    "lower_loss_model": b if db > 0 else a})
D = pd.DataFrame(dm_rows); D["p_holm"] = holm(D["p"])
allt = pd.concat([T, D], ignore_index=True); allt.to_csv("d2_all_tests.csv", index=False)

rep("\nPaired t-tests across folds (A - B; Holm-adjusted over all pairs)")
rep(T.drop(columns=["test"]).round(4).to_string(index=False))
rep("\nDiebold-Mariano on the held-out test errors (squared loss; DM>0: A has larger loss; n = test rows)")
rep(D[["A", "B", "n_test_rows", "stat", "p", "p_holm", "stat_NeweyWest12", "p_NeweyWest12", "lower_loss_model"]].round(4).to_string(index=False))
rep("\nCAUTION: expanding-window folds overlap and five folds give little power, so the t-tests are indicative; the DM tests use the full test set but its errors are autocorrelated (see the Newey-West columns).")

# ---------------- Table 8 + predictions, from this same run
t8 = results_df.copy()
t8["R2_fold_mean_recomputed"] = [fold_r2[k].mean() if k in fold_r2 else np.nan for k in [next((n for n, tn in table_name.items() if tn == m), m) for m in t8["Model"]]]
t8.to_csv("table8_numbers.csv", index=False)
pd.DataFrame({"Timestamp": test_df["Timestamp"].values, "y_true": yte, **{f"pred_{k}": v for k, v in preds.items()}}).to_csv("d2_test_predictions.csv", index=False)
open("d2_tests_report.txt", "w", encoding="utf-8").write("\n".join(_rep))
rep("\nSaved: table8_numbers.csv, d2_fold_r2.csv, d2_all_tests.csv, d2_test_predictions.csv, d2_run_meta.json, d2_tests_report.txt")

Loading Dataset
             Timestamp  deviceId  payload_id  air_temperature  humidity  CO2  \
0  2024-09-25 19:25:00         2           1            26.30     64.26  455   
1  2024-09-25 19:30:00         2           2            26.29     64.29  452   
2  2024-09-25 19:35:00         2           3            26.30     64.32  442   
3  2024-09-25 19:40:00         2           4            26.29     64.34  442   
4  2024-09-25 19:45:00         2           5            26.29     64.44  437   

    pm1   pm10  pm2_5  air_pressure  
0  61.5  119.9  111.1        100994  
1  60.2  119.8  111.0        100995  
2  63.1  125.5  114.3        100998  
3  62.1  123.0  112.9        100998  
4  62.7  124.2  114.4        101000  

Dataset Shape : (52259, 10)

Detected Columns
---------------------------
CO2        : CO2
Temperature: air_temperature
Humidity   : humidity
Occupancy  : None
PM2.5      : pm2_5
TVOC       : None

Creating Time Features...

Creating Sensor Interaction Features...

Creating

2026-10-25 19:42:39,604 INFO Part 1 completed successfully.



Creating EMA Features...

Dataset after Feature Engineering
(52234, 45)

Training Samples : 41787
Testing Samples  : 10447
PART 2 - PREPROCESSING

Removing Outliers...
PART 3

Linear Regression
MAE  : 19.98
RMSE : 47.60
R²   : 0.9893
CV   : 0.9900 ± 0.0037

Decision Tree
MAE  : 24.17
RMSE : 59.53
R²   : 0.9833
CV   : 0.9837 ± 0.0033
PART 4

Random Forest
MAE  : 19.91
RMSE : 47.42
R²   : 0.9894
CV   : 0.9889 ± 0.0024
Fitting 5 folds for each of 8 candidates, totalling 40 fits

Gradient Boosting
MAE  : 43.36
RMSE : 72.19
R²   : 0.9754
CV   : 0.9803 ± 0.0064

XGBoost
MAE  : 23.41
RMSE : 51.93
R²   : 0.9873
CV   : 0.9807 ± 0.0053
FINAL MODEL COMPARISON
               Model         MAE        RMSE        R2   CV Mean    CV Std
0      Random Forest   19.911904   47.420110  0.989384  0.988916  0.002372
1  Linear Regression   19.977632   47.599803  0.989304  0.990045  0.003667
2            XGBoost   23.407584   51.928559  0.987270  0.980655  0.005321
3  Lag-1 Persistence   23.870106   57.8150

In [2]:
from scipy import stats
import numpy as np
import pandas as pd

# Example structure: collecting fold-wise R² scores for each model across 5 CV splits
# (Replace fold_r2_* with actual cross_val_score outputs with cv=tscv, scoring="r2")

models_cv_scores = {
    "Linear Regression": cross_val_score(linear_pipeline, X_train, y_train, cv=tscv, scoring="r2"),
    "Baseline RF": cross_val_score(rf_pipeline, X_train, y_train, cv=tscv, scoring="r2"),
    "Decision Tree": cross_val_score(decision_tree_pipeline, X_train, y_train, cv=tscv, scoring="r2"),
    "Gradient Boosting": cross_val_score(gb_pipeline, X_train, y_train, cv=tscv, scoring="r2"),
}

if 'xgb_pipeline' in locals():
    models_cv_scores["XGBoost"] = cross_val_score(xgb_pipeline, X_train, y_train, cv=tscv, scoring="r2")

# Perform pairwise comparisons against Random Forest (Baseline)
baseline_name = "Baseline RF"
baseline_scores = models_cv_scores[baseline_name]

significance_results = []
for model_name, scores in models_cv_scores.items():
    if model_name == baseline_name:
        continue
    # Paired t-test
    t_stat, p_val = stats.ttest_rel(baseline_scores, scores)
    # Wilcoxon signed-rank test as non-parametric alternative
    try:
        w_stat, w_pval = stats.wilcoxon(baseline_scores, scores)
    except Exception:
        w_stat, w_pval = np.nan, np.nan
        
    significance_results.append({
        "Model_A": baseline_name,
        "Model_B": model_name,
        "Mean_Diff": baseline_scores.mean() - scores.mean(),
        "t_statistic": t_stat,
        "p_value_ttest": p_val,
        "wilcoxon_stat": w_stat,
        "p_value_wilcoxon": w_pval,
        "Significant_0.05": p_val < 0.05
    })

sig_df = pd.DataFrame(significance_results)
print(sig_df)

# Export significance tests to CSV
sig_df.to_csv("Model_Significance_Tests.csv", index=False)

NameError: name 'cross_val_score' is not defined

In [2]:
# New #

In [3]:
# ============================================================
# IoT Indoor Air Quality Prediction
# FINAL THESIS VERSION (Leak-Free + Ablation + Audits)
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import os
import re
import random
import logging
import joblib

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

# ============================================================
# SKLEARN
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.impute import SimpleImputer

from sklearn.model_selection import (
    TimeSeriesSplit,
    RandomizedSearchCV,
    cross_val_score
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.covariance import EllipticEnvelope
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)
from sklearn.feature_selection import SelectKBest, f_regression

# ============================================================
# OPTIONAL LIBRARIES
# ============================================================

try:
    from xgboost import XGBRegressor
    XGB_AVAILABLE = True
except:
    XGB_AVAILABLE = False

try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    SARIMA_AVAILABLE = True
except:
    SARIMA_AVAILABLE = False

try:
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping
    from sklearn.preprocessing import MinMaxScaler
    LSTM_AVAILABLE = True
except:
    LSTM_AVAILABLE = False

# ============================================================
# CONFIGURATION
# ============================================================

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s"
)

logger = logging.getLogger(__name__)

# ============================================================
# DATA PATH
# ============================================================

DATA_PATH = os.environ.get("IAQ_D2_PATH", r"C:\Users\Mero\OneDrive\Documents\data collection\device2_air_quality_2024-09-25_to_2025-04-07.csv")

# ============================================================
# LOAD DATA
# ============================================================

print("="*80)
print("Loading Dataset")
print("="*80)

df = pd.read_csv(DATA_PATH)
print(df.head())
print()
print("Dataset Shape :", df.shape)

# ============================================================
# TIMESTAMP
# ============================================================

df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df = df.sort_values("Timestamp").reset_index(drop=True)

# ============================================================
# ROBUST COLUMN DETECTION
# ============================================================

def normalize(name):
    name = name.lower()
    name = re.sub(r"[^a-z0-9]", "", name)
    return name

normalized = {c: normalize(c) for c in df.columns}

def detect_column(include, exclude=None):
    for original, norm in normalized.items():
        if exclude:
            if any(e in norm for e in exclude):
                continue
        if all(i in norm for i in include):
            return original
    return None

CO2_COL  = detect_column(["co2"])
TEMP_COL = detect_column(["temperature"])
HUM_COL  = detect_column(["humidity"])
OCC_COL  = detect_column(["occupancy"])
PM25_COL = detect_column(["pm2","5"], exclude=["ppm"])
TVOC_COL = detect_column(["tvoc"])

print()
print("Detected Columns")
print("---------------------------")
print("CO2        :", CO2_COL)
print("Temperature:", TEMP_COL)
print("Humidity   :", HUM_COL)
print("Occupancy  :", OCC_COL)
print("PM2.5      :", PM25_COL)
print("TVOC       :", TVOC_COL)

# ============================================================
# TIME FEATURES
# ============================================================

print()
print("Creating Time Features...")

df["Hour"] = df["Timestamp"].dt.hour
df["Day"] = df["Timestamp"].dt.day
df["Weekday"] = df["Timestamp"].dt.dayofweek
df["Month"] = df["Timestamp"].dt.month
df["Weekend"] = (df["Weekday"] >= 5).astype(int)

df["Hour_sin"] = np.sin(2 * np.pi * df["Hour"] / 24)
df["Hour_cos"] = np.cos(2 * np.pi * df["Hour"] / 24)

# ============================================================
# SENSOR INTERACTION FEATURES
# ============================================================

print()
print("Creating Sensor Interaction Features...")

if TEMP_COL and HUM_COL:
    df["Temp_Humidity"] = df[TEMP_COL] * df[HUM_COL]
if OCC_COL:
    df["Occupancy_CO2"] = df[OCC_COL]
if PM25_COL:
    df["PM25_CO2"] = df[PM25_COL]
if TVOC_COL:
    df["TVOC_CO2"] = df[TVOC_COL]

# ============================================================
# LEAK-FREE LAG FEATURES
# ============================================================

print()
print("Creating Lag Features...")

LAGS = [1, 2, 3, 6, 12, 24]
for lag in LAGS:
    df[f"CO2_Lag_{lag}"] = df[CO2_COL].shift(lag)

# ============================================================
# LEAK-FREE ROLLING FEATURES
# ============================================================

print()
print("Creating Rolling Features...")

history = df[CO2_COL].shift(1)
WINDOWS = [3, 6, 12, 24]

for window in WINDOWS:
    df[f"RollingMean_{window}"] = history.rolling(window).mean()
    df[f"RollingStd_{window}"] = history.rolling(window).std()
    df[f"RollingMin_{window}"] = history.rolling(window).min()
    df[f"RollingMax_{window}"] = history.rolling(window).max()

# ============================================================
# EXPONENTIAL MOVING AVERAGES
# ============================================================

print()
print("Creating EMA Features...")

df["EMA_6"] = history.ewm(span=6).mean()
df["EMA_12"] = history.ewm(span=12).mean()
df["EMA_24"] = history.ewm(span=24).mean()

# ============================================================
# FUTURE TARGET
# ============================================================

FORECAST_HORIZON = 1
df["Target"] = df[CO2_COL].shift(-FORECAST_HORIZON)

# ============================================================
# REMOVE NAN
# ============================================================

df = df.dropna().reset_index(drop=True)

print()
print("Dataset after Feature Engineering")
print(df.shape)

# ============================================================
# CHRONOLOGICAL TRAIN TEST SPLIT
# ============================================================

split_index = int(len(df) * 0.80)
train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

print()
print("="*80)
print("Training Samples :", len(train_df))
print("Testing Samples  :", len(test_df))
print("="*80)

logger.info("Part 1 completed successfully.")

# ============================================================
# PART 2: DATA PREPROCESSING
# ============================================================

print("="*80)
print("PART 2 - PREPROCESSING")
print("="*80)

X_train = train_df.drop(columns=["Target", "Timestamp"])
y_train = train_df["Target"]
X_test = test_df.drop(columns=["Target", "Timestamp"])
y_test = test_df["Target"]

# Remove Outliers (Training Data Only)
print()
print("Removing Outliers...")
numeric_cols = X_train.select_dtypes(include=np.number).columns
outlier_detector = EllipticEnvelope(contamination=0.01, random_state=RANDOM_STATE)
mask = outlier_detector.fit_predict(X_train[numeric_cols]) == 1

X_train = X_train.loc[mask].reset_index(drop=True)
y_train = y_train.loc[mask].reset_index(drop=True)

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

linear_selector = SelectKBest(score_func=f_regression, k=min(25, len(numeric_features)))
tscv = TimeSeriesSplit(n_splits=5)

results = []

def evaluate_model(model_name, model):
    model.fit(X_train, y_train)
    prediction = model.predict(X_test)

    mae = mean_absolute_error(y_test, prediction)
    rmse = np.sqrt(mean_squared_error(y_test, prediction))
    r2 = r2_score(y_test, prediction)

    cv_scores = cross_val_score(model, X_train, y_train, cv=tscv, scoring="r2", n_jobs=-1)

    print()
    print("="*60)
    print(model_name)
    print("="*60)
    print(f"MAE  : {mae:.2f}")
    print(f"RMSE : {rmse:.2f}")
    print(f"R²   : {r2:.4f}")
    print(f"CV   : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

    results.append({
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "CV Mean": cv_scores.mean(),
        "CV Std": cv_scores.std()
    })
    return model

# ============================================================
# PART 3: BASELINE + LINEAR REGRESSION + DECISION TREE
# ============================================================

print("=" * 80)
print("PART 3")
print("=" * 80)

linear_pipeline = Pipeline([
    ("prep", preprocessor),
    ("select", linear_selector),
    ("model", LinearRegression())
])

decision_tree_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", DecisionTreeRegressor(max_depth=12, min_samples_leaf=5, random_state=RANDOM_STATE))
])

# Lag-1 Persistence Baseline
lag_prediction = test_df[CO2_COL].values
lag_actual = y_test.values
lag_mae = mean_absolute_error(lag_actual, lag_prediction)
lag_rmse = np.sqrt(mean_squared_error(lag_actual, lag_prediction))
lag_r2 = r2_score(lag_actual, lag_prediction)

results.append({
    "Model": "Lag-1 Persistence",
    "MAE": lag_mae,
    "RMSE": lag_rmse,
    "R2": lag_r2,
    "CV Mean": np.nan,
    "CV Std": np.nan
})

linear_model = evaluate_model("Linear Regression", linear_pipeline)
decision_tree_model = evaluate_model("Decision Tree", decision_tree_pipeline)

# ============================================================
# PART 4: TREE-BASED MODELS
# ============================================================

print("="*80)
print("PART 4")
print("="*80)

rf_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestRegressor(n_estimators=300, max_depth=15, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1))
])

random_forest_model = evaluate_model("Random Forest", rf_pipeline)

# Tuned RF
parameter_grid = {
    "model__n_estimators": [150, 250],
    "model__max_depth": [10, 15, None],
    "model__min_samples_leaf": [1, 2],
    "model__min_samples_split": [2, 5],
    "model__max_features": ["sqrt"]
}

rf_base = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1))
])

tuned_rf = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=parameter_grid,
    n_iter=8,
    cv=tscv,
    scoring="r2",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)
tuned_rf.fit(X_train, y_train)
best_rf = tuned_rf.best_estimator_

pred_trf = best_rf.predict(X_test)
trf_mae = mean_absolute_error(y_test, pred_trf)
trf_rmse = np.sqrt(mean_squared_error(y_test, pred_trf))
trf_r2 = r2_score(y_test, pred_trf)

results.append({
    "Model": "Tuned RF",
    "MAE": trf_mae,
    "RMSE": trf_rmse,
    "R2": trf_r2,
    "CV Mean": tuned_rf.best_score_,
    "CV Std": np.nan
})

gb_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=5, subsample=0.8, random_state=RANDOM_STATE))
])
gradient_boosting_model = evaluate_model("Gradient Boosting", gb_pipeline)

if XGB_AVAILABLE:
    xgb_pipeline = Pipeline([
        ("prep", preprocessor),
        ("model", XGBRegressor(objective="reg:squarederror", n_estimators=300, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE, n_jobs=-1))
    ])
    xgb_model = evaluate_model("XGBoost", xgb_pipeline)

# ============================================================
# PART 5: SARIMA + LSTM
# ============================================================

if SARIMA_AVAILABLE:
    # Rolling ONE-STEP ARIMA(2,1,2): parameters fitted on TRAIN only, then filtered through the test
    # set with the observed history -> genuine 1-step-ahead forecasts (same task as the other models).
    # (The old forecast(steps=len(test)) was a static multi-step forecast = flat line, not comparable.)
    train_series = train_df[CO2_COL].reset_index(drop=True)
    test_series = test_df["Target"].reset_index(drop=True)
    sarima_fit = SARIMAX(train_series, order=(2,1,2), seasonal_order=(0,0,0,0),
                         enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    full_series = pd.concat([train_series, test_df[CO2_COL].reset_index(drop=True), test_series.iloc[[-1]]], ignore_index=True)
    applied = sarima_fit.apply(full_series, refit=False)          # keep training parameters
    sarima_prediction = applied.get_prediction(start=len(train_series) + 1, dynamic=False).predicted_mean.values
    assert len(sarima_prediction) == len(test_series)

    s_mae = mean_absolute_error(test_series, sarima_prediction)
    s_rmse = np.sqrt(mean_squared_error(test_series, sarima_prediction))
    s_r2 = r2_score(test_series, sarima_prediction)
    print(f"ARIMA(2,1,2) rolling 1-step | MAE {s_mae:.2f} RMSE {s_rmse:.2f} R2 {s_r2:.4f}")

    results.append({
        "Model": "ARIMA(2,1,2) rolling 1-step",
        "MAE": s_mae,
        "RMSE": s_rmse,
        "R2": s_r2,
        "CV Mean": np.nan,
        "CV Std": np.nan
    })

if LSTM_AVAILABLE:
    X_train_lstm = preprocessor.fit_transform(X_train)
    X_test_lstm = preprocessor.transform(X_test)
    if hasattr(X_train_lstm, "toarray"):
        X_train_lstm = X_train_lstm.toarray()
        X_test_lstm = X_test_lstm.toarray()

    feature_scaler = MinMaxScaler()
    target_scaler = MinMaxScaler()
    X_train_scaled = feature_scaler.fit_transform(X_train_lstm)
    X_test_scaled = feature_scaler.transform(X_test_lstm)
    y_train_scaled = target_scaler.fit_transform(y_train.values.reshape(-1,1))
    y_test_scaled = target_scaler.transform(y_test.values.reshape(-1,1))

    WINDOW_LSTM = 24
    LSTM_OFFSET = WINDOW_LSTM - 1          # first test row the LSTM predicts
    def make_sequences(X, y, window):
        # window rows i..i+W-1 predicts the Target OF ITS LAST ROW (= CO2[t+1]) -> one step ahead
        xs, ys = [], []
        for i in range(len(X) - window + 1):
            xs.append(X[i:i+window])
            ys.append(y[i+window-1])
        return np.array(xs), np.array(ys)

    X_seq_train, y_seq_train = make_sequences(X_train_scaled, y_train_scaled, WINDOW_LSTM)
    X_seq_test, y_seq_test = make_sequences(X_test_scaled, y_test_scaled, WINDOW_LSTM)

    lstm_model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(WINDOW_LSTM, X_seq_train.shape[2])),
        Dropout(0.2),
        LSTM(32),
        Dense(1)
    ])
    lstm_model.compile(optimizer="adam", loss="mse")
    early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    
    lstm_model.fit(X_seq_train, y_seq_train, validation_split=0.10, epochs=30, batch_size=32, callbacks=[early_stop], verbose=0)
    
    pred_scaled = lstm_model.predict(X_seq_test, verbose=0)
    lstm_pred = target_scaler.inverse_transform(pred_scaled).flatten()
    lstm_actual = target_scaler.inverse_transform(y_seq_test).flatten()

    l_mae = mean_absolute_error(lstm_actual, lstm_pred)
    l_rmse = np.sqrt(mean_squared_error(lstm_actual, lstm_pred))
    l_r2 = r2_score(lstm_actual, lstm_pred)

    results.append({
        "Model": "LSTM",
        "MAE": l_mae,
        "RMSE": l_rmse,
        "R2": l_r2,
        "CV Mean": np.nan,
        "CV Std": np.nan
    })

# ============================================================
# PART 6: FINAL EVALUATION & EXPORTS
# ============================================================

print("="*90)
print("FINAL MODEL COMPARISON")
print("="*90)

results_df = pd.DataFrame(results).sort_values(by="R2", ascending=False).reset_index(drop=True)
print(results_df)

results_df.to_csv("Model_Comparison.csv", index=False)

# ============================================================
# PART 7: ABLATION STUDY
#   (a) no engineered lags/rolling/EMA but CURRENT CO2 kept  (CO2(t) is itself the persistence feature)
#   (b) no CO2 history at all (current CO2 removed too)  <- the true "no autocorrelation" ablation
# ============================================================

print("="*80)
print("PART 7 - ABLATION STUDY")
print("="*80)

hist_kw = ["lag", "rolling", "ema"]
no_engineered = [c for c in numeric_features if not any(k in c.lower() for k in hist_kw)]
no_history = [c for c in no_engineered if c != CO2_COL]

def _abl_pipe(cols):
    prep = ColumnTransformer([("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")),
                                                    ("scaler", StandardScaler())]), cols)], remainder="drop")
    return Pipeline([("prep", prep),
                     ("model", RandomForestRegressor(n_estimators=300, max_depth=15, min_samples_leaf=2,
                                                     random_state=RANDOM_STATE, n_jobs=-1))])

print(f"All numeric: {len(numeric_features)} | no engineered lags/rolling/EMA: {len(no_engineered)} | no CO2 history at all: {len(no_history)}")
evaluate_model("Ablation RF (no engineered lags; current CO2 kept)", _abl_pipe(no_engineered))
evaluate_model("Ablation RF (no CO2 history at all)", _abl_pipe(no_history))


# ============================================================
# PART 9: SIGNIFICANCE TESTS FROM THE SAME RUN AS TABLE 8
#   - per-fold R2 for every model (same TimeSeriesSplit as Part 3), paired t-tests (df = 4), Holm-adjusted
#   - Diebold-Mariano tests on the SAME test-set errors that produced Table 8 (h = 1, HLN correction; Newey-West check)
#   - provenance: data file hash, library versions, row counts, random seed
#   Nothing here is typed in by hand: every number is computed from the fitted models in this session.
#   Sign convention: for "A vs B", t = mean(R2_A - R2_B) / SE   and   DM > 0 means A has the LARGER loss (B is better).
# ============================================================
import json, hashlib, platform, itertools
import sklearn, scipy
from scipy import stats
from sklearn.base import clone

_rep = []
def rep(*a):
    s = " ".join(str(x) for x in a); print(s); _rep.append(s)

rep("="*90); rep("PART 9 - TESTS FROM THE SAME RUN AS TABLE 8"); rep("="*90)

# ---------------- provenance
def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()
meta = {"run_time": pd.Timestamp.now().isoformat(), "data_file": os.path.basename(DATA_PATH), "data_sha256": _sha256(DATA_PATH),
        "n_train_after_outlier_removal": int(len(X_train)), "n_test": int(len(X_test)), "random_state": RANDOM_STATE,
        "python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "scikit_learn": sklearn.__version__,
        "scipy": scipy.__version__, "tuned_rf_best_params": {k: (None if v is None else (v if isinstance(v, (int, float, str)) else str(v))) for k, v in tuned_rf.best_params_.items()}}
try:
    import xgboost; meta["xgboost"] = xgboost.__version__
except Exception: pass
json.dump(meta, open("d2_run_meta.json", "w"), indent=2); rep("provenance:", json.dumps({k: meta[k] for k in ("data_file", "data_sha256", "n_train_after_outlier_removal", "n_test")}))

# ---------------- fitted models from this session
fitted = {"Linear Regression": linear_model, "Decision Tree": decision_tree_model, "Random Forest (baseline)": random_forest_model,
          "Tuned RF": best_rf, "Gradient Boosting": gradient_boosting_model}
if XGB_AVAILABLE: fitted["XGBoost"] = xgb_model
table_name = {"Linear Regression": "Linear Regression", "Decision Tree": "Decision Tree", "Random Forest (baseline)": "Random Forest",
              "Tuned RF": "Tuned RF", "Gradient Boosting": "Gradient Boosting", "XGBoost": "XGBoost"}

# ---------------- A) per-fold R2 (identical folds: tscv on the outlier-filtered training set)
fold_r2 = {n: cross_val_score(clone(m), X_train, y_train, cv=tscv, scoring="r2", n_jobs=-1) for n, m in fitted.items()}
fold_df = pd.DataFrame(fold_r2); fold_df.index = [f"fold {i+1}" for i in range(len(fold_df))]
rep("\nPer-fold R2 (5 expanding-window folds)"); rep(fold_df.round(4).to_string())
chk = results_df.set_index("Model")["CV Mean"]
for n in fold_r2:
    logged = chk.get(table_name[n], np.nan)
    rep(f"  reproduction check {n:26s} fold mean = {fold_r2[n].mean():.6f} | CV Mean in Table 8 = {logged:.6f} | "
        + ("OK" if abs(fold_r2[n].mean() - logged) < 1e-6 else "MISMATCH - investigate"))
fold_df.to_csv("d2_fold_r2.csv")

def holm(p):
    p = np.asarray(p, float); order = np.argsort(p); m = len(p); adj = np.empty(m); run = 0.0
    for rank, i in enumerate(order): run = max(run, (m - rank) * p[i]); adj[i] = min(1.0, run)
    return adj

rows = []
for a, b in itertools.combinations(fold_r2, 2):
    t, p = stats.ttest_rel(fold_r2[a], fold_r2[b])
    try: w = stats.wilcoxon(fold_r2[a], fold_r2[b]).pvalue
    except Exception: w = np.nan
    rows.append({"test": "paired t (5 folds, df=4)", "A": a, "B": b, "mean_diff_R2_A_minus_B": float((fold_r2[a] - fold_r2[b]).mean()),
                 "stat": t, "p": p, "wilcoxon_p": w, "folds_where_A_better": int((fold_r2[a] > fold_r2[b]).sum())})
T = pd.DataFrame(rows); T["p_holm"] = holm(T["p"])

# ---------------- B) Diebold-Mariano on the test errors behind Table 8
yte = y_test.values
preds = {n: m.predict(X_test) for n, m in fitted.items()}
preds["Lag-1 Persistence (rule)"] = test_df[CO2_COL].values
offset = 0
if "lstm_pred" in globals():
    offset = LSTM_OFFSET                       # LSTM predicts test rows LSTM_OFFSET .. end only
    lstm_full = np.full(len(yte), np.nan); lstm_full[offset:] = lstm_pred; preds["LSTM"] = lstm_full
    if not np.allclose(lstm_actual, yte[offset:], atol=1e-3):
        rep("WARNING: LSTM targets are not aligned with y_test; LSTM is left out of the Diebold-Mariano tests"); preds.pop("LSTM")
if "sarima_prediction" in globals():
    preds["ARIMA(2,1,2) rolling 1-step"] = np.asarray(sarima_prediction)
def _dm(e1, e2, loss="se", lags=12):
    d = (e1**2 - e2**2) if loss == "se" else (np.abs(e1) - np.abs(e2)); n = len(d); db = d.mean(); dc = d - db
    g0 = np.mean(dc*dc); hln = db/np.sqrt(g0/n) * np.sqrt((n - 1)/n)                       # h = 1 -> long-run variance = gamma0
    lr = g0 + 2*sum((1 - k/(lags+1))*np.mean(dc[k:]*dc[:-k]) for k in range(1, lags+1)); nw = db/np.sqrt(lr/n)
    return hln, 2*stats.t.sf(abs(hln), n-1), nw, 2*stats.norm.sf(abs(nw)), db, n
dm_rows = []
for a, b in itertools.combinations(preds, 2):
    ok = ~(np.isnan(preds[a]) | np.isnan(preds[b])); ea, eb = (yte - preds[a])[ok], (yte - preds[b])[ok]
    s, p, snw, pnw, db, n = _dm(ea, eb, "se"); sa, pa, *_ = _dm(ea, eb, "ae")
    dm_rows.append({"test": "Diebold-Mariano (h=1, HLN)", "A": a, "B": b, "n_test_rows": n, "mean_sq_loss_diff_A_minus_B": db,
                    "stat": s, "p": p, "stat_NeweyWest12": snw, "p_NeweyWest12": pnw, "stat_abs_loss": sa, "p_abs_loss": pa,
                    "lower_loss_model": b if db > 0 else a})
D = pd.DataFrame(dm_rows); D["p_holm"] = holm(D["p"])
allt = pd.concat([T, D], ignore_index=True); allt.to_csv("d2_all_tests.csv", index=False)

rep("\nPaired t-tests across folds (A - B; Holm-adjusted over all pairs)")
rep(T.drop(columns=["test"]).round(4).to_string(index=False))
rep("\nDiebold-Mariano on the held-out test errors (squared loss; DM>0: A has larger loss; n = test rows)")
rep(D[["A", "B", "n_test_rows", "stat", "p", "p_holm", "stat_NeweyWest12", "p_NeweyWest12", "lower_loss_model"]].round(4).to_string(index=False))
rep("\nCAUTION: expanding-window folds overlap and five folds give little power, so the t-tests are indicative; the DM tests use the full test set but its errors are autocorrelated (see the Newey-West columns).")

# ---------------- Table 8 + predictions, from this same run
t8 = results_df.copy()
t8["R2_fold_mean_recomputed"] = [fold_r2[k].mean() if k in fold_r2 else np.nan for k in [next((n for n, tn in table_name.items() if tn == m), m) for m in t8["Model"]]]
t8.to_csv("table8_numbers.csv", index=False)
pd.DataFrame({"Timestamp": test_df["Timestamp"].values, "y_true": yte, **{f"pred_{k}": v for k, v in preds.items()}}).to_csv("d2_test_predictions.csv", index=False)
open("d2_tests_report.txt", "w", encoding="utf-8").write("\n".join(_rep))
rep("\nSaved: table8_numbers.csv, d2_fold_r2.csv, d2_all_tests.csv, d2_test_predictions.csv, d2_run_meta.json, d2_tests_report.txt")

Loading Dataset
             Timestamp  deviceId  payload_id  air_temperature  humidity  CO2  \
0  2024-09-25 19:25:00         2           1            26.30     64.26  455   
1  2024-09-25 19:30:00         2           2            26.29     64.29  452   
2  2024-09-25 19:35:00         2           3            26.30     64.32  442   
3  2024-09-25 19:40:00         2           4            26.29     64.34  442   
4  2024-09-25 19:45:00         2           5            26.29     64.44  437   

    pm1   pm10  pm2_5  air_pressure  
0  61.5  119.9  111.1        100994  
1  60.2  119.8  111.0        100995  
2  63.1  125.5  114.3        100998  
3  62.1  123.0  112.9        100998  
4  62.7  124.2  114.4        101000  

Dataset Shape : (52259, 10)


2026-10-30 21:50:41,290 INFO Part 1 completed successfully.



Detected Columns
---------------------------
CO2        : CO2
Temperature: air_temperature
Humidity   : humidity
Occupancy  : None
PM2.5      : pm2_5
TVOC       : None

Creating Time Features...

Creating Sensor Interaction Features...

Creating Lag Features...

Creating Rolling Features...

Creating EMA Features...

Dataset after Feature Engineering
(52234, 45)

Training Samples : 41787
Testing Samples  : 10447
PART 2 - PREPROCESSING

Removing Outliers...
PART 3

Linear Regression
MAE  : 19.98
RMSE : 47.60
R²   : 0.9893
CV   : 0.9900 ± 0.0037

Decision Tree
MAE  : 24.17
RMSE : 59.53
R²   : 0.9833
CV   : 0.9837 ± 0.0033
PART 4

Random Forest
MAE  : 19.91
RMSE : 47.42
R²   : 0.9894
CV   : 0.9889 ± 0.0024
Fitting 5 folds for each of 8 candidates, totalling 40 fits

Gradient Boosting
MAE  : 43.36
RMSE : 72.19
R²   : 0.9754
CV   : 0.9803 ± 0.0064

XGBoost
MAE  : 23.41
RMSE : 51.93
R²   : 0.9873
CV   : 0.9807 ± 0.0053
ARIMA(2,1,2) rolling 1-step | MAE 20.15 RMSE 49.37 R2 0.9885
FINAL MODEL

In [4]:
##########3

In [6]:
# ============================================================
# PART 7: CORRECTED THREE-STAGE ABLATION STUDY
# ============================================================

print("=" * 80)
print("PART 7 - THREE-STAGE ABLATION STUDY")
print("=" * 80)

# ------------------------------------------------------------
# 1. Identify feature groups
# ------------------------------------------------------------

co2_history_features = [
    c for c in numeric_features
    if any(k in c.lower() for k in ["lag", "rolling", "ema"])
]

current_co2_feature = (
    [CO2_COL] if CO2_COL in numeric_features else []
)

# A1: sensor + calendar + interaction features
# Exclude current CO2 and all engineered CO2-history features.
A1_features = [
    c for c in numeric_features
    if c not in current_co2_feature
    and c not in co2_history_features
]

# A2: A1 + current CO2
A2_features = A1_features + current_co2_feature

# A3: A2 + lag, rolling and EMA features
A3_features = A2_features + co2_history_features

# ------------------------------------------------------------
# 2. Verify feature groups
# ------------------------------------------------------------

print("\nFeature configuration:")
print("A1 features:", len(A1_features))
print("A2 features:", len(A2_features))
print("A3 features:", len(A3_features))

print("\nCO2 current feature:", current_co2_feature)
print("CO2 history features:", len(co2_history_features))

if len(A1_features) == 0:
    raise ValueError("A1 contains no features. Check numeric_features.")

if len(A2_features) <= len(A1_features):
    print("WARNING: Current CO2 was not added to A2.")

if len(A3_features) <= len(A2_features):
    print("WARNING: No CO2 lag/rolling/EMA features were added to A3.")

# ------------------------------------------------------------
# 3. Fixed Random Forest configuration
# ------------------------------------------------------------

def create_ablation_model(feature_list):

    ablation_preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler())
                ]),
                feature_list
            )
        ],
        remainder="drop"
    )

    return Pipeline([
        ("prep", ablation_preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=300,
                max_depth=15,
                min_samples_leaf=2,
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
        )
    ])

# ------------------------------------------------------------
# 4. Evaluate each configuration
# ------------------------------------------------------------

ablation_results = []

def run_ablation(name, features):

    model = create_ablation_model(features)

    model.fit(X_train[features], y_train)

    prediction = model.predict(X_test[features])

    mae = mean_absolute_error(y_test, prediction)

    rmse = np.sqrt(
        mean_squared_error(y_test, prediction)
    )

    r2 = r2_score(y_test, prediction)

    # Same chronological CV strategy for all configurations
    cv_scores = cross_val_score(
        model,
        X_train[features],
        y_train,
        cv=tscv,
        scoring="r2",
        n_jobs=-1
    )

    result = {
        "Configuration": name,
        "Feature_Count": len(features),
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae,
        "CV_R2_Mean": cv_scores.mean(),
        "CV_R2_Std": cv_scores.std()
    }

    ablation_results.append(result)

    print("\n" + "-" * 60)
    print(name)
    print("-" * 60)
    print("Feature count:", len(features))
    print(f"R2   : {r2:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"MAE  : {mae:.4f}")
    print(
        f"CV R2: {cv_scores.mean():.4f} "
        f"+/- {cv_scores.std():.4f}"
    )

    return result

# ------------------------------------------------------------
# 5. Run A1, A2 and A3
# ------------------------------------------------------------

A1 = run_ablation(
    "A1 - Sensor and Calendar Features",
    A1_features
)

A2 = run_ablation(
    "A2 - A1 + Current CO2",
    A2_features
)

A3 = run_ablation(
    "A3 - A2 + CO2 Historical Features",
    A3_features
)

# ------------------------------------------------------------
# 6. Calculate R2 contributions
# ------------------------------------------------------------

ablation_df = pd.DataFrame(ablation_results)

r2_A1 = A1["R2"]
r2_A2 = A2["R2"]
r2_A3 = A3["R2"]

ablation_df["Delta_R2"] = [
    np.nan,
    r2_A2 - r2_A1,
    r2_A3 - r2_A2
]

ablation_df["Delta_RMSE"] = [
    np.nan,
    A2["RMSE"] - A1["RMSE"],
    A3["RMSE"] - A2["RMSE"]
]

# ------------------------------------------------------------
# 7. Save results
# ------------------------------------------------------------

ablation_df.to_csv(
    "Corrected_Ablation_Results.csv",
    index=False
)

print("\n" + "=" * 80)
print("FINAL ABLATION RESULTS")
print("=" * 80)

print(
    ablation_df.round(4).to_string(index=False)
)

print("\nR2 contribution of current CO2:")
print(f"A2 - A1 = {r2_A2 - r2_A1:+.4f}")

print("\nR2 contribution of historical CO2 features:")
print(f"A3 - A2 = {r2_A3 - r2_A2:+.4f}")

print("\nSaved: Corrected_Ablation_Results.csv")

print("=" * 80)

PART 7 - THREE-STAGE ABLATION STUDY

Feature configuration:
A1 features: 17
A2 features: 18
A3 features: 43

CO2 current feature: ['CO2']
CO2 history features: 25

------------------------------------------------------------
A1 - Sensor and Calendar Features
------------------------------------------------------------
Feature count: 17
R2   : 0.1218
RMSE : 431.3069
MAE  : 271.0227
CV R2: -0.8727 +/- 0.6985

------------------------------------------------------------
A2 - A1 + Current CO2
------------------------------------------------------------
Feature count: 18
R2   : 0.9803
RMSE : 64.6049
MAE  : 29.7400
CV R2: 0.9810 +/- 0.0054

------------------------------------------------------------
A3 - A2 + CO2 Historical Features
------------------------------------------------------------
Feature count: 43
R2   : 0.9894
RMSE : 47.4643
MAE  : 19.9268
CV R2: 0.9889 +/- 0.0024

FINAL ABLATION RESULTS
                    Configuration  Feature_Count     R2     RMSE      MAE  CV_R2_Mean  CV_